# PDF → EPUB engine — local dev iteration

Thin driver over `packages/converter` (same code paths as the CLI and Celery worker).

**Fixtures:** use synthetic PDFs from tests (`reportlab`) or a **local path on your machine**. Do not commit commercial books or licensed PDFs to git.

### One-time setup (not installed in production Docker images)

```bash
cd /path/to/refolio
python3 -m venv .venv && source .venv/bin/activate
pip install -e "packages/converter[dev,notebook]"
python -m ipykernel install --user --name refolio-converter --display-name "refolio-converter"
```

Optional env vars: `CONVERTER_PARSER`, `CONVERTER_OCR_ENGINE`, `EPUBCHECK_JAR`.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

from converter.adapters.factory import get_parser
from converter.contracts import ParseContext
from converter.editorial import apply_editorial_passes
from converter.ir.models import BlockType, DocumentIR
from converter.pipeline import ConversionPipeline

# --- Configure input (edit this) ---
REPO_ROOT = Path("..").resolve()  # notebooks/ -> repo root
PDF_PATH = REPO_ROOT / "notebooks" / ".local" / "sample.pdf"  # create under .local (gitignored)
OUTPUT_DIR = REPO_ROOT / "notebooks" / "output" / PDF_PATH.stem
LANGUAGE_HINT = "es"
MAX_PAGES: int | None = None  # e.g. 5 for quick runs

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PDF_PATH = Path(PDF_PATH).expanduser().resolve()
assert PDF_PATH.is_file(), f"PDF not found: {PDF_PATH} — place a file or generate one in the next cell"
print("PDF:", PDF_PATH)
print("Output:", OUTPUT_DIR)

### Optional: generate a synthetic PDF (no external fixture)

In [ ]:
from reportlab.lib.pagesizes import letter
from reportlab.pdfgen import canvas

local_dir = REPO_ROOT / "notebooks" / ".local"
local_dir.mkdir(parents=True, exist_ok=True)
synthetic = local_dir / "synthetic_dev.pdf"

c = canvas.Canvas(str(synthetic), pagesize=letter)
c.setFont("Helvetica", 12)
c.drawString(72, 720, "Capítulo 1")
c.drawString(72, 700, "Este párrafo sintético sirve para probar el motor.")
c.showPage()
c.drawString(72, 720, "Segunda página con texto de continuación")
c.save()

PDF_PATH = synthetic.resolve()
OUTPUT_DIR = REPO_ROOT / "notebooks" / "output" / PDF_PATH.stem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Using synthetic PDF:", PDF_PATH)

## 1. Parse → raw IR (routing + line extraction)

In [ ]:
parser = get_parser()
ctx = ParseContext(source_path=PDF_PATH, language_hint=LANGUAGE_HINT, max_pages=MAX_PAGES)
raw_ir: DocumentIR = parser.parse(ctx)

print("Parser:", raw_ir.engine_trace.get("parser"))
print("Schema:", raw_ir.schema_version)
print("Pages:", len(raw_ir.pages), "| LINE blocks:", sum(1 for b in raw_ir.blocks if b.block_type == BlockType.LINE))
print()
print("Page diagnostics:")
for d in raw_ir.page_diagnostics:
    print(f"  p{d.page_index}: chars={d.char_count} alnum={d.alnum_ratio:.2f} route={d.route.value}")

In [ ]:
print("Warnings (parse):")
for w in raw_ir.warnings:
    loc = f" p{w.page_index}" if w.page_index is not None else ""
    print(f"  [{w.code}]{loc} {w.message}")

## 2. Editorial passes (running matter → paragraphs → chapters)

In [ ]:
edited_ir = apply_editorial_passes(raw_ir)

from collections import Counter

counts = Counter(b.block_type.value for b in edited_ir.blocks)
print("Block types after editorial:", dict(counts))
print("Editorial trace:", edited_ir.engine_trace.get("editorial"))
print()
print("New warnings after editorial:")
new_warn = edited_ir.warnings[len(raw_ir.warnings):]
for w in new_warn:
    print(f"  [{w.code}] {w.message}")

In [ ]:
def block_summary(doc: DocumentIR, limit: int = 12) -> list[dict]:
    rows = []
    for b in doc.sorted_blocks()[:limit]:
        rows.append(
            {
                "order": b.reading_order,
                "type": b.block_type.value,
                "text": b.current_text[:80],
                "pages": b.provenance.page_indices,
                "method": b.provenance.extraction_method.value,
                "changes": len(b.change_history),
            }
        )
    return rows

print("Raw (first blocks):")
for row in block_summary(raw_ir):
    print(row)
print()
print("Edited (first blocks):")
for row in block_summary(edited_ir):
    print(row)

In [ ]:
print("Provenance / change history (edited blocks with transforms):")
for b in edited_ir.sorted_blocks():
    if not b.change_history:
        continue
    print(f"--- {b.id} ({b.block_type.value}) ---")
    print(" text:", b.current_text[:120])
    for ch in b.change_history:
        print(f"   {ch.op.value}: {ch.reason} | evidence={ch.evidence}")

## 3. Full pipeline → EPUB + quality report (production path)

In [ ]:
pipeline_out = OUTPUT_DIR / "pipeline_run"
result = ConversionPipeline().run(
    source_pdf=PDF_PATH,
    output_dir=pipeline_out,
    language_hint=LANGUAGE_HINT,
    max_pages=MAX_PAGES,
    run_epubcheck_validation=True,
)

print("Engine version:", result.engine_version)
print("EPUB:", result.epub_path, "exists=", result.epub_path.is_file())
print("IR JSON:", result.ir_path)
print("Quality report keys:", list(result.quality_report.to_dict().keys()))
validation = result.quality_report.to_dict().get("technical_validation")
if validation:
    print("EPUBCheck:", validation)

In [ ]:
# Persist a copy of the final IR for diffing outside the notebook
snapshot = OUTPUT_DIR / "document.ir.snapshot.json"
snapshot.write_text(
    json.dumps(result.document_ir.model_dump(mode="json"), indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("Wrote", snapshot)